# Verification of the RG3 innovation candidate
Derive confusion counts from actual predictions, not copied metrics. Verify explicit RH targets and measured sensor mapping, causal original delta extraction, train/test timestamp and block exclusions, future-only training restriction, and baseline preservation.
This verifies the exploratory candidate's reported scope. Does NOT establish new-production generalization, competition score or drying causation.
Target criterion: whole TP>=5 FP<=6, plus forward evaluation and documented limitations.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
p=pd.read_csv(tables/'rg3_innovation_seed_sensitivity_predictions.csv',parse_dates=['TimeStamp'])
p=p.loc[p.model.eq('three_seed_mean')].copy()
b=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
b=b.loc[b.model.eq('max_union')].copy()
assert not p.duplicated(['scheme','TimeStamp']).any()
z=p.merge(b[['scheme','TimeStamp','actual','predicted_train_fpr1']],on=['scheme','TimeStamp'],validate='one_to_one',suffixes=('_new','_old'))
assert len(z)==1023 and (z.actual_new==z.actual_old).all()
assert (z.baseline_prediction.astype(bool)==z.predicted_train_fpr1).all()
assert (z.prediction>=z.baseline_prediction).all()
d=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d.loc[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.str.contains('가스',na=False))].copy()
rh=pd.read_csv(root/'data/audit/rg3_rh_labeled_with_timestamp.csv',parse_dates=['TimeStamp'])
rh=rh.loc[rh.PassOrFail.eq(0)|rh.Reason.str.contains('가스',na=False)]
mapping=d.merge(rh[['TimeStamp','PassOrFail','Reason','Barrel_Temperature_5','Max_Injection_Pressure']],on='TimeStamp',validate='one_to_one',suffixes=('_features','_RH'))
assert len(mapping)==583 and (mapping.PassOrFail_features==mapping.PassOrFail_RH).all()
assert (mapping.Reason_features==mapping.Reason_RH).all()
assert np.allclose(mapping.Barrel_Temperature_5_features,mapping.Barrel_Temperature_5_RH)
assert np.allclose(mapping.Max_Injection_Pressure_features,mapping.Max_Injection_Pressure_RH)
# Inspect the exact original-history extraction used by the candidate.
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
exec(''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source']).split('for feature in features:')[0])
observed['audit_delta_barrel5']=observed.groupby('history_segment').Barrel_Temperature_5.diff()
history_keys=['EQUIP_CD','product','TimeStamp']
hist=observed.loc[observed['product'].eq('RG3'),history_keys+['audit_delta_barrel5']]
current=d[history_keys+['Barrel_Temperature_5__delta1']].merge(hist,on=history_keys,validate='one_to_one')
assert len(current)==583 and np.allclose(current.Barrel_Temperature_5__delta1,current.audit_delta_barrel5,equal_nan=True)
inner=pd.read_csv(tables/'rg3_rank_fusion_inner_scores.csv',parse_dates=['TimeStamp'])
blocklist=d.comparison_block.drop_duplicates().tolist()
for (scheme,block),q in p.groupby(['scheme','block']):
    ref=inner.loc[inner.scheme.eq(scheme)&inner.heldout_block.eq(block)]
    assert not ref.block.eq(block).any()
    assert set(ref.TimeStamp).isdisjoint(set(q.TimeStamp))
    if scheme=='past_blocks_only':
        assert set(ref.block).issubset(set(blocklist[:blocklist.index(block)]))
    assert np.all(q.prediction.eq(0)|q.baseline_prediction.eq(1)|((q.expert_score>q.expert_threshold)&(q.current_barrel5_delta>q.delta_threshold)))
    assert np.allclose(q.current_barrel5_delta.to_numpy(),d.set_index('TimeStamp').loc[q.TimeStamp,'Barrel_Temperature_5__delta1'].to_numpy(),equal_nan=True)
rows=[]
for scheme,g in p.groupby('scheme',sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    old=b.loc[b.scheme.eq(scheme)]
    tn0,fp0,fn0,tp0=confusion_matrix(old.actual,old.predicted_train_fpr1,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,shots=len(g),gas=int(g.actual.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),baseline_TP=int(tp0),baseline_FP=int(fp0),new_false_alarms=int(((g.prediction==1)&(g.baseline_prediction==0)&(g.actual==0)).sum())))
audit=pd.DataFrame(rows)
whole=audit.loc[audit.scheme.eq('leave_one_block_out')].iloc[0]
future=audit.loc[audit.scheme.eq('past_blocks_only')].iloc[0]
assert whole.shots==583 and whole.gas==17 and whole.TP==5 and whole.FP==6 and whole.FN==12 and whole.TN==560
assert future.shots==440 and future.gas==13 and future.TP==3 and future.FP==4 and future.FN==10 and future.TN==423
assert (audit.new_false_alarms==0).all()
audit.to_csv(tables/'rg3_innovation_candidate_verification.csv',index=False,encoding='utf-8-sig')
gas=p.loc[p.scheme.eq('leave_one_block_out')&p.actual.eq(1),['TimeStamp','block','baseline_prediction','prediction','expert_score','expert_threshold','current_barrel5_delta','delta_threshold']]
gas.to_csv(tables/'rg3_innovation_candidate_all17.csv',index=False,encoding='utf-8-sig')
print(audit.to_string(index=False))
print('RH target mapping, causal delta, all fold exclusions and all 1023 baseline decisions verified.')
print('CAVEAT: test-informed feature ideation; inner feature selection tied with added TP=0; no independent new production test.')


## Executed results
Executed audit passes: RH labels/sensors, equipment-product-timestamp causal delta,1023 original decisions preserved, fold exclusions and past-only restrictions. Recomputed whole5/6/12/560 and forward3/4/10/423. No new false alarms. Does not prove independent generalization.
See reports/rg3_current_innovation_confirmation.md and reports/rg3_remaining12_cold_signature_review.md.